In [ ]:
import os
import mne
import mne_bids
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import re
from mne_bids import BIDSPath, read_raw_bids
%matplotlib widget


In [ ]:
stimtrack_bids_root = r"C:\Users\Laura\Documents\PhD\Soundbrain lab\EAM\data-bids\python"
trigger_bids_root = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids"

In [ ]:
sub_folders = os.listdir(stimtrack_bids_root)
onset_differences = pd.DataFrame(columns=['subject', 'stimtrack_onset', 'trigger_onset', 'difference_ms'])

In [ ]:
for sub in sub_folders:

    sub_num = int(sub.split('-')[1])
    stimtrack_path = os.path.join(stimtrack_bids_root, sub)
    trigger_path = os.path.join(trigger_bids_root, sub, 'eeg')

    stimtrack_files = [f for f in os.listdir(stimtrack_path) if f.endswith('events.tsv')]
    trigger_files = [f for f in os.listdir(trigger_path) if f.endswith('events.tsv')]

    for i, file in enumerate(stimtrack_files):
        stimtrack_events = pd.read_csv(os.path.join(stimtrack_path, file), sep='\t')
        trigger_events = pd.read_csv(os.path.join(trigger_path, trigger_files[i]), sep='\t')
        onset_diff = stimtrack_events['onset'] - trigger_events['onset']
        onset_differences = pd.concat([onset_differences, pd.DataFrame({
            'subject': [sub],
            'stimtrack_onset': [stimtrack_events['onset'].values[0]],
            'trigger_onset': [trigger_events['onset'].values[0]],
            'difference_ms': [onset_diff.values[0]]
        })], ignore_index=True)

In [ ]:
plt.figure()
plt.hist(onset_differences['difference_ms'], bins=100)
plt.xlabel('Stimtrack Onset - Trigger Onset (ms)')
plt.ylabel('Count')
plt.title('Distribution of Onset Differences Between StimTrack and Trigger Events')
plt.show()

In [ ]:
num_greater_zero = np.sum(onset_differences['difference_ms'] != 0)
total_differences = len(onset_differences)
print(num_greater_zero, total_differences, num_greater_zero / total_differences)

num_greater_pointone = np.sum(onset_differences['difference_ms'] > .1)
print(num_greater_pointone, total_differences, num_greater_pointone / total_differences)

num_less_pointone = np.sum(onset_differences['difference_ms'] < -.1)
print(num_less_pointone, total_differences, num_less_pointone / total_differences)

In [ ]:
plt.figure()
plt.hist(onset_differences['difference_ms'].iloc[np.where(onset_differences['difference_ms'] < 0.1)[0]], bins=100)
plt.xlabel('Stimtrack Onset - Trigger Onset (ms)')
plt.ylabel('Count')
plt.title('Distribution of Onset Differences Between StimTrack and Trigger Events')
plt.show()